# 03 - Feature Engineering
Create time-based, categorical, and derived features from cleaned data. Merge BoB features. Save engineered dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
%matplotlib inline

from config import PROCESSED_DATA_DIR, DATE_COLS, TIER_MAP
from src.util import load_dataframe, save_dataframe

## 3.1 Load cleaned data

In [ ]:
df = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'retention_cleaned.csv'))
bob = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'bob_cleaned.csv'))
print(f'Starting shape: {df.shape}')

## 3.2 Time-based features

In [ ]:
# parse date columns
for col in DATE_COLS:
    if col in df.columns:
        df[col + '_parsed'] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
        parsed_count = df[col + '_parsed'].notna().sum()
        print(f'{col}: parsed {parsed_count:,} / {len(df):,}')

In [ ]:
# compute time differences
if 'Case Creation Date_parsed' in df.columns and 'Expected Pull Date_parsed' in df.columns:
    df['days_to_pull'] = (df['Expected Pull Date_parsed'] - df['Case Creation Date_parsed']).dt.days
    print(f'days_to_pull: mean={df["days_to_pull"].mean():.0f}, median={df["days_to_pull"].median():.0f}')

if 'Case Creation Date_parsed' in df.columns and 'Registered Date_parsed' in df.columns:
    df['days_reg_to_case'] = (df['Case Creation Date_parsed'] - df['Registered Date_parsed']).dt.days
    print(f'days_reg_to_case: mean={df["days_reg_to_case"].mean():.0f}, median={df["days_reg_to_case"].median():.0f}')

if 'Case Creation Date_parsed' in df.columns and 'Agreement End Date_parsed' in df.columns:
    df['months_to_agreement_end'] = (
        (df['Agreement End Date_parsed'] - df['Case Creation Date_parsed']).dt.days / 30.44
    ).clip(lower=-120, upper=120)
    print(f'months_to_agreement_end: mean={df["months_to_agreement_end"].mean():.1f}')

# calendar features
if 'Case Creation Date_parsed' in df.columns:
    df['case_month'] = df['Case Creation Date_parsed'].dt.month
    df['case_year'] = df['Case Creation Date_parsed'].dt.year
    df['case_dayofweek'] = df['Case Creation Date_parsed'].dt.dayofweek
    print('Calendar features created: case_month, case_year, case_dayofweek')

## 3.3 Categorical encoding

In [ ]:
# Case Type -> binary flags
if 'Case Type' in df.columns:
    df['is_cancellation'] = (df['Case Type'] == 'Cancellation').astype(int)
    df['is_risk_case'] = (df['Case Type'] == 'Risk').astype(int)
    print(f'is_cancellation: {df["is_cancellation"].sum():,}')
    print(f'is_risk_case: {df["is_risk_case"].sum():,}')

# Pull Type -> flags
if 'Pull Type' in df.columns:
    df['is_full_pull'] = (df['Pull Type'] == 'Full').astype(int)
    df['has_pull_type'] = (df['Pull Type'] != 'None').astype(int)
    print(f'is_full_pull: {df["is_full_pull"].sum():,}')

# Risk -> one-hot encoding
if 'Risk' in df.columns:
    risk_dummies = pd.get_dummies(df['Risk'], prefix='risk', dummy_na=False)
    df = pd.concat([df, risk_dummies], axis=1)
    print(f'Risk dummies created: {list(risk_dummies.columns)}')

In [ ]:
# Case Origin -> proactive vs reactive
if 'Case Origin' in df.columns:
    proactive = ['Account Manager', 'Proactive Prevention', 'Branch Manager',
                 'Service Manager', 'Customer Service Manager', 'Head of Customer Services']
    reactive = ['Notice in Writing', 'Customer Email', 'Customer Call',
                'Customer Letter', 'Email', 'Phone', 'Fax', 'Web']
    df['origin_proactive'] = df['Case Origin'].isin(proactive).astype(int)
    df['origin_reactive'] = df['Case Origin'].isin(reactive).astype(int)
    print(f'Proactive: {df["origin_proactive"].sum():,}, Reactive: {df["origin_reactive"].sum():,}')

# Customer Tier -> ordinal numeric
if 'Customer Tier' in df.columns:
    df['tier_numeric'] = df['Customer Tier'].map(TIER_MAP)
    print(f'tier_numeric created. Unique values: {sorted(df["tier_numeric"].dropna().unique())}')

# Churn reason -> dummies
if 'churn_reason_group' in df.columns:
    reason_dummies = pd.get_dummies(df['churn_reason_group'], prefix='reason')
    df = pd.concat([df, reason_dummies], axis=1)
    print(f'Churn reason dummies: {list(reason_dummies.columns)}')

## 3.4 Derived numeric features

In [ ]:
# VAN per machine and per contract
if 'VAN' in df.columns and 'Machines' in df.columns:
    df['van_per_machine'] = df['VAN'] / df['Machines'].replace(0, np.nan)
    print(f'van_per_machine: mean={df["van_per_machine"].mean():.2f}')

if 'VAN' in df.columns and 'Number of Contracts' in df.columns:
    df['van_per_contract'] = df['VAN'] / df['Number of Contracts'].replace(0, np.nan)
    print(f'van_per_contract: mean={df["van_per_contract"].mean():.2f}')

# repair and overdue features
if 'Number Of Repair Cases' in df.columns:
    df['repair_rate'] = df['Number Of Repair Cases'].fillna(0)
    df['has_repairs'] = (df['repair_rate'] > 0).astype(int)
    print(f'has_repairs: {df["has_repairs"].sum():,}')

if 'Number of OverdueServices' in df.columns:
    df['overdue_rate'] = df['Number of OverdueServices'].fillna(0)
    df['has_overdue'] = (df['overdue_rate'] > 0).astype(int)
    print(f'has_overdue: {df["has_overdue"].sum():,}')

# pull VAN ratio
if 'Pull VAN' in df.columns and 'VAN' in df.columns:
    df['pull_van_ratio'] = df['Pull VAN'] / df['VAN'].replace(0, np.nan)
    print(f'pull_van_ratio: mean={df["pull_van_ratio"].mean():.4f}')

## 3.5 Merge BoB features

In [ ]:
# aggregate BoB by account
bob_agg = bob.groupby('account_number').agg(
    bob_total_revenue=('total_bob', 'sum'),
    bob_mean_revenue=('total_bob', 'mean'),
    bob_product_revenue=('product_bob', 'sum'),
    bob_fee_revenue=('fee_bob', 'sum'),
    bob_num_agreements=('agreement_number', 'nunique'),
    bob_num_products=('product_name', 'nunique'),
    bob_lines_of_business=('line_of_business', 'nunique'),
    bob_avg_service_interval=('service_interval', 'mean'),
    bob_avg_unit_amount=('unit_amount', 'mean'),
    bob_pct_auto_renewal=('renewal_type', lambda x: (x == 'Automatic Renewal').mean()),
    bob_has_machine_services=('line_of_business', lambda x: int('Machine Services' in x.values)),
    bob_has_auto_waste=('line_of_business', lambda x: int('Auto waste' in x.values)),
).reset_index()

print(f'BoB aggregated: {len(bob_agg):,} accounts')

# merge into retention
df = df.merge(bob_agg, left_on='Customer Account Number', right_on='account_number', how='left')
df['has_bob_data'] = df['account_number'].notna().astype(int)
print(f'BoB match rate: {df["has_bob_data"].mean():.1%}')

## 3.6 Handle remaining nulls

In [ ]:
# fill remaining numeric nulls with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
null_filled = 0
for col in numeric_cols:
    n_null = df[col].isnull().sum()
    if n_null > 0:
        df[col] = df[col].fillna(df[col].median())
        null_filled += 1

print(f'Filled nulls in {null_filled} numeric columns')
print(f'Remaining nulls: {df.select_dtypes(include=[np.number]).isnull().sum().sum()}')

## 3.7 Save engineered data

In [ ]:
print(f'Final shape: {df.shape}')
print(f'Numeric features: {len(df.select_dtypes(include=[np.number]).columns)}')
print(f'\nNew columns created:')

# show new columns (not in original retention)
original_cols = load_dataframe(os.path.join(PROCESSED_DATA_DIR, 'retention_cleaned.csv')).columns.tolist()
new_cols = [c for c in df.columns if c not in original_cols]
for c in new_cols:
    print(f'  {c}')

save_dataframe(df, os.path.join(PROCESSED_DATA_DIR, 'df_engineered.csv'), 'engineered features')